In [401]:
from langchain.agents import create_agent
from langchain_classic.tools import retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import StructuredTool
# langchain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool, ToolRuntime, BaseTool

# langchain_core imports
from langchain_core.documents import  Document
from langchain_core.prompts import ChatPromptTemplate, format_document
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableLambda

# langgraph Imports
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# langchain_community imports
from langchain_community.document_loaders import (
    DirectoryLoader, UnstructuredMarkdownLoader, PythonLoader, TextLoader
)
from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker, EmbeddingsFilter
from langchain_community.cross_encoders import  HuggingFaceCrossEncoder
from langchain_text_splitters import (
    PythonCodeTextSplitter, TextSplitter, MarkdownTextSplitter, RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)

from langchain_chroma import Chroma
from langgraph.types import Command
from openai.types.conversations import input_text_content

from environ import Environment, Models

from typing_extensions import TypedDict
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from itertools import batched, chain
from collections.abc import Sequence

import typing
import hashlib
import pathlib
import uuid

In [69]:
DOCUMENT_DIR: pathlib.Path = pathlib.Path("../docs/docs").resolve()
if not DOCUMENT_DIR.exists():
    raise NotADirectoryError("Directory doesn't exists:", DOCUMENT_DIR)

In [140]:
class AIProviders:
    llm: ChatOpenAI = ChatOpenAI(base_url=Environment.url, api_key=Environment.key,
                                 model=Models.Gpt_5_4_nano, temperature=0.4,
    )
    embeddings: OpenAIEmbeddings = OpenAIEmbeddings( base_url=Environment.url, api_key=Environment.key,
                                                     model=Models.Embedding_Small,
    )

class VectorStore:
    store: Chroma = Chroma(collection_name="embeddings", persist_directory="./chroma_db",
                                 embedding_function=AIProviders.embeddings)

In [45]:
loader: DirectoryLoader = DirectoryLoader(path=str(DOCUMENT_DIR), glob="**/*.md",
                                          loader_cls=TextLoader)
documents: list[Document] = loader.load()

In [110]:
def split_document() -> typing.List[Document]:
    chunks: typing.List[Document] = []
    header: typing.List[typing.Tuple[str, str]] = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3")
    ]
    header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=header)
    char_splitter = RecursiveCharacterTextSplitter(separators= [
        "\n", "\n###", "\n##", "\n#", " ", ""], chunk_size=500, chunk_overlap=50)

    for idx, document in enumerate(documents):
        source = pathlib.Path(document.metadata["source"]).resolve()
        document_id = hashlib.sha256(f"{source}:{idx}:{document.page_content}".encode()).hexdigest()
        base_metadata: typing.Dict[str, typing.Hashable | str] = {
            **document.metadata,
            "document_id": document_id,
            "filename": source.name,
            "stem": source.stem,
            "extension": source.suffix,
        }
        sections = header_splitter.split_text(text=document.page_content)
        global_chunk_index: int = 0

        for section in sections:
            section.metadata.update({**base_metadata, **section.metadata})

            for chunk in char_splitter.split_documents(documents=[section]):
                chunk_id = hashlib.sha256(f"{document_id}:{global_chunk_index}".encode()).hexdigest()
                global_chunk_index += 1

                chunk.metadata.update({"chunk_id": chunk_id})
                chunks.append(chunk)

    return chunks

In [111]:
BATCH_SIZE: int = 1000

def ingest_document(chunks: typing.List[Document], store: Chroma):
    ids: typing.List[typing.Any] = [doc.metadata["chunk_id"] for doc in chunks]
    for docs_batch, ids_batch in zip(batched(chunks, BATCH_SIZE), batched(ids, BATCH_SIZE)):
        store.add_documents(documents=list(docs_batch), ids=list(ids_batch))

In [112]:
def generate_embeddings() -> Chroma:
    chunks: typing.List[Document] = split_document()
    store: Chroma = Chroma(collection_name="embeddings", persist_directory="./chroma_db",
                                 embedding_function=AIProviders.embeddings)
    ingest_document(chunks, store)
    return store

In [215]:
vectorstore = generate_embeddings()

In [216]:
CHUNKS: list[Document] = split_document()

In [243]:
def create_ensembler(search_type: str = "mmr", search_kwargs: dict[str, typing.Any] | None = None,
                     weights: list[float] | None= None) -> EnsembleRetriever:
    chunks: typing.List[Document] = CHUNKS

    if weights is None:
        weights = [0.6, 0.4]

    if search_kwargs is None:
        search_kwargs = {"k": 10, "lambda_mul": 0.5, "fetch_k": 30}

    base_retriever = VectorStore.store.as_retriever(search_type=search_type, search_kwargs=search_kwargs)
    bm25_retriever = BM25Retriever.from_documents(documents=chunks)

    ensembler: EnsembleRetriever = EnsembleRetriever(retrievers=[base_retriever, bm25_retriever], weights=weights)
    return ensembler

In [244]:
def create_contextual_ensembler(search_type: str = "mmr", search_kwargs: dict[str, typing.Any] | None = None, similarity_threshold = 0.3, weights: list[float] | None = None) -> ContextualCompressionRetriever:
    chunks: typing.List[Document] = CHUNKS
    if search_kwargs is None:
        search_kwargs = {"k": 10, "lambda_mul": 0.5, "fetch_k": 30}

    if weights is None:
        weights = [0.6, 0.4]

    base_retriever = VectorStore.store.as_retriever(search_type=search_type, search_kwargs=search_kwargs)
    bm25_retriever = BM25Retriever.from_documents(documents=chunks)
    ensembler: EnsembleRetriever = EnsembleRetriever(retrievers=[base_retriever, bm25_retriever], weights=weights)

    compressor = EmbeddingsFilter(embeddings=AIProviders.embeddings, similarity_threshold=similarity_threshold)
    compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=ensembler)
    return compression_retriever

In [220]:
format_docs = RunnableLambda(
    lambda docs: "\n\n".join([doc.page_content for doc in docs])
)

In [228]:
def create_rag_llm():
    ensembler = create_ensembler()
    prompt = ChatPromptTemplate.from_template(
         """
Answer the question based on the following context:
{context}

Question: {question}

Answer: Make sure to answer in a concise manner. If you are not sure then just say "I don't know"
"""
    )

    rag_chain = (
        {"context": ensembler | format_docs, "question": RunnablePassthrough()}
        | prompt
        | AIProviders.llm
        | StrOutputParser()
    )
    return rag_chain

In [374]:
config = RunnableConfig(configurable={"thread_id": str(uuid.uuid4())})
rag_config = RunnableConfig(configurable={"thread_id": str(uuid.uuid4())})
checkpoints = InMemorySaver()

In [230]:
chain = create_rag_llm()
image = chain.get_graph().draw_mermaid_png()
with open("archiecture.png", "wb") as f:
    f.write(image)

In [375]:
class RetrievalResult(BaseModel):
    # id: str
    summary: str
    sources: list[str]

In [376]:
class CustomToolKit:
    def __init__(self, configurable: RunnableConfig):
        self.config = configurable
        self.WORKSPACE_DIR: pathlib.Path = (pathlib.Path.cwd() / "workspace").resolve()

    def rag_search(self, question: str):
        """Search aquilia Documentation for given question"""
        ensembler = create_contextual_ensembler(
            search_type="mmr", search_kwargs={"k": 20, "lambda_mul": 0.5, "fetch_k": 30},
            similarity_threshold=0.3, weights=[0.6, 0.4]
        )
        # structured_llm_response = AIProviders.llm.with_structured_output(RetrievalResult)
        prompt = ChatPromptTemplate.from_template(
            """
    You are a documentation retrieval specialist.

    Your task is to search the aquilia documentation and return only the information that is relevant to the user's request.

    Rules:

    * Focus on finding implementation details, APIs, classes, functions, decorators, middleware, configuration, and examples.
    * Prefer exact matches over semantic guesses.
    * Return relevant code examples when available.
    * Include source file names if they are available.
    * Do not answer the user's request.
    * Do not generate code that is not present in the retrieved documentation.
    * Do not explain concepts beyond what is contained in the documentation.
    * Return only documentation context that will help another agent complete the task.

    User Request:
    {question}

    Retrieved Context:
    {context}

    Return a concise but complete summary of the relevant documentation.
            """
        )

        rag_chain = (
            {"context": ensembler | format_docs, "question": RunnablePassthrough()}
            | prompt
            | AIProviders.llm.with_structured_output(RetrievalResult)
            # | AIProviders.llm
            # | StrOutputParser()
        )

        llm_response = rag_chain.invoke(input=question, config=self.config)
        return llm_response

    def safe_path(self, path: str) -> pathlib.Path:
        path = path.strip()

        if path.startswith("workspace/"):
            path = path[len("workspace/"):]

        elif path == "workspace":
            path = "."

        resolved = (self.WORKSPACE_DIR / path).resolve()

        if not str(resolved).startswith(str(self.WORKSPACE_DIR)):
            raise ValueError("Path outside workspace")

        return resolved

    def read_file(self, path: str) -> str:
        """
        Read a file and return its contents.
        """

        safe_path = self.safe_path(path)

        try:
            with open(safe_path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()

        except Exception as e:
            return f"Error reading file: {e}"

    def write_file(self, path: str, content: str) -> str:
        """
        Write content to a file.

        Path must be relative to the workspace root.
        Examples:
        - main.py
        - src/app.py
        - app/services/user.py

        Do NOT include 'workspace/' in paths.
        """

        safe_path = self.safe_path(path)

        try:
            safe_path.parent.mkdir(parents=True, exist_ok=True)

            with open(safe_path, "w", encoding="utf-8") as f:
                f.write(content)

            return f"Successfully wrote {len(content)} characters."

        except Exception as e:
            return f"Error writing file: {e}"

    def append_file(self, path: str, content: str) -> str:
        """
        Append content to an existing file.
        """

        safe_path = self.safe_path(path)

        try:
            safe_path.parent.mkdir(parents=True, exist_ok=True)

            with open(safe_path, "a", encoding="utf-8") as f:
                f.write(content)

            return f"Appended {len(content)} characters."

        except Exception as e:
            return f"Error appending file: {e}"

    def edit_file(
        self,
        path: str,
        old_text: str,
        new_text: str,
    ) -> str:
        """
        Replace text inside a file.
        """

        safe_path = self.safe_path(path)

        try:
            with open(safe_path, "r", encoding="utf-8") as f:
                content = f.read()

            occurrences = content.count(old_text)

            if occurrences == 0:
                return "Text not found."

            updated = content.replace(old_text, new_text)

            with open(safe_path, "w", encoding="utf-8") as f:
                f.write(updated)

            return f"Replaced {occurrences} occurrence(s)."

        except Exception as e:
            return f"Error editing file: {e}"


    def replace_lines(
        self,
        path: str,
        start_line: int,
        end_line: int,
        replacement: str,
    ) -> str:
        """
        Replace a range of lines.
        """

        safe_path = self.safe_path(path)

        try:
            with open(safe_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            lines[start_line - 1:end_line] = [
                replacement + "\n"
            ]

            with open(safe_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return (
                f"Replaced lines "
                f"{start_line}-{end_line}"
            )

        except Exception as e:
            return f"Error: {e}"

    def grep(self, path: str, query: str) -> list[dict]:
        """
        Search for a string inside a file.

        Returns matching lines with line numbers.
        """
        safe_path = self.safe_path(path)

        results = []

        try:
            with open(safe_path, "r", encoding="utf-8", errors="ignore") as f:
                for lineno, line in enumerate(f, start=1):
                    if query.lower() in line.lower():
                        results.append({
                            "line": lineno,
                            "content": line.rstrip()
                        })
        except Exception as e:
            return [{"error": str(e)}]

        return results

    def search(self, directory: str, query: str) -> list[dict]:
        """
        Recursively search all files for a string.
        """

        base = self.safe_path(directory)
        matches = []

        for file in pathlib.Path(base).rglob("*"):
            if not file.is_file():
                continue

            try:
                with open(file, "r", encoding="utf-8", errors="ignore") as f:
                    for lineno, line in enumerate(f, start=1):
                        if query.lower() in line.lower():
                            matches.append({
                                "file": str(file),
                                "line": lineno,
                                "content": line.rstrip()
                            })
            except Exception:
                pass

        return matches[:100]

    def glob(self, pattern: str) -> list[str]:
        """
        Find files matching a glob pattern.
        Example:
            *.py
            **/*.md
        """

        root = self.safe_path(".")
        return [
            str(p)
            for p in pathlib.Path(root).glob(pattern)
        ]

    def mkdir(self, path: str) -> str:
        """Make directory"""
        safe_path = self.safe_path(path)

        safe_path.mkdir(
            parents=True,
            exist_ok=True
        )

        return "Directory created"

    def ls(self, path: str = ".") -> list[dict]:
        """
        List files and directories.
        """

        safe_path = self.safe_path(path)

        results = []

        for item in safe_path.iterdir():
            results.append({
                "name": item.name,
                "type": "dir" if item.is_dir() else "file"
            })

        return sorted(
            results,
            key=lambda x: (x["type"] == "file", x["name"])
        )
    # @tool()

In [377]:
toolkit = CustomToolKit(rag_config)
toolbox: Sequence[StructuredTool] =  [
    StructuredTool.from_function(toolkit.glob, name="glob"),
    StructuredTool.from_function(toolkit.grep, name="grep"),
    StructuredTool.from_function(toolkit.search, name="search"),
    StructuredTool.from_function(toolkit.read_file, name="read_file"),
    StructuredTool.from_function(toolkit.rag_search, name="rag_search"),
    StructuredTool.from_function(toolkit.replace_lines, name="replace_lines"),
    StructuredTool.from_function(toolkit.write_file, name="write_file"),
    StructuredTool.from_function(toolkit.edit_file, name="edit_file"),
    StructuredTool.from_function(toolkit.append_file, name="append_file"),
    StructuredTool.from_function(toolkit.ls, name="ls"),
    StructuredTool.from_function(toolkit.mkdir, name="mkdir"),
]
# print(toolkit.rag_search("What happens when AuthGuard optional=False?"))

In [378]:
for tool in toolbox:
    print(type(tool))
    print(tool.name)

<class 'langchain_core.tools.structured.StructuredTool'>
glob
<class 'langchain_core.tools.structured.StructuredTool'>
grep
<class 'langchain_core.tools.structured.StructuredTool'>
search
<class 'langchain_core.tools.structured.StructuredTool'>
read_file
<class 'langchain_core.tools.structured.StructuredTool'>
rag_search
<class 'langchain_core.tools.structured.StructuredTool'>
replace_lines
<class 'langchain_core.tools.structured.StructuredTool'>
write_file
<class 'langchain_core.tools.structured.StructuredTool'>
edit_file
<class 'langchain_core.tools.structured.StructuredTool'>
append_file
<class 'langchain_core.tools.structured.StructuredTool'>
ls
<class 'langchain_core.tools.structured.StructuredTool'>
mkdir


In [238]:
def evaluate_rag(dataset: list[dict[str, str]], ensembler: EnsembleRetriever | ContextualCompressionRetriever)-> dict[str, float]:
    hit_at_1 = 0
    hit_at_3 = 0
    hit_at_5 = 0

    mrr: float = 0.0
    total = len(dataset)

    for item in dataset:
        # print(item.get("source"))
        docs: Sequence[Document] = ensembler.invoke(input=item["question"])
        expected_doc = item["expected_doc"]
        rank = None

        for idx, doc in enumerate(docs, start=1):
            if doc.metadata.get("source") == expected_doc:
                rank = idx
                break

        if rank is not None:
            if rank <= 1:
                hit_at_1 += 1

            if rank <= 3:
                hit_at_3 += 1

            if rank <= 5:
                hit_at_5 += 1

            mrr += 1 / rank

    return {
        "hit@1": hit_at_1 / total,
        "hit@3": hit_at_3 / total,
        "hit@5": hit_at_5 / total,
        "mrr": mrr / total,
    }

In [239]:
def doc(relative_path: str) -> str:
    return str((DOCUMENT_DIR / relative_path).resolve())

eval_dataset: list[dict[str, str]] = [
    {"question": question, "expected_doc": doc(expected_doc)}
    for question, expected_doc in [
        ("What order does ConfigLoader.load merge configuration sources?", "configuration.md"),
        ("Which environment variable normalizes production to prod?", "configuration.md"),
        ("What does AQ_AUTH__TOKENS__SECRET_KEY map to?", "configuration.md"),
        ("Which config file format is removed and should be migrated away from?", "configuration.md"),
        ("Which file contains the class-based AquilaConfig definition?", "configuration.md"),
        ("Which module contains ConfigLoader and namespace access?", "configuration.md"),
        ("Which environment variable is used as the runtime mode by AquiliaRuntime and the entrypoint?", "configuration.md"),
        ("Which variables are used as signing secret fallbacks?", "configuration.md"),
        ("Which example is the CRUD workspace?", "examples.md"),
        ("Which commands are shown as the operational flow for examples?", "examples.md"),
        ("Where do tests live in each example?", "examples.md"),
        ("What does the CRUD example wire together?", "examples.md"),
        ("What does the auth starter add besides DI and routing?", "examples.md"),
        ("What fields are listed in the Manifest Pattern example?", "examples.md"),
        ("What framework types does the HTTP Controller Pattern use?", "examples.md"),
        ("What does the WebSocket Pattern use for acknowledgements?", "examples.md"),
        ("What does the Background Task Pattern decorate with task priority and schedule?", "examples.md"),
        ("What workspace integration enables auth system wiring?", "auth-methods.md"),
        ("What happens when AuthGuard optional=False?", "auth-methods.md"),
        ("What header does ApiKeyGuard use?", "auth-methods.md"),
        ("What does AuthzGuard evaluate?", "auth-methods.md"),
        ("What are RequireSessionAuthGuard and RequireTokenAuthGuard for?", "auth-methods.md"),
        ("How do you override a class-level guard for one route?", "auth-methods.md"),
        ("What are the common access levels in clearance?", "auth-methods.md"),
        ("What does require_auth=True mean in middleware-level auth?", "auth-methods.md"),
        ("What does aq init workspace create?", "cli-reference.md"),
        ("Which option makes aq init workspace minimal?", "cli-reference.md"),
        ("Which command adds a new module to the workspace?", "cli-reference.md"),
        ("Which option sets dependencies for aq add module?", "cli-reference.md"),
        ("Which command generates a new controller?", "cli-reference.md"),
        ("Which option makes aq generate controller include lifecycle hooks?", "cli-reference.md"),
        ("Which command validates workspace manifests?", "cli-reference.md"),
        ("Which option makes aq validate emit JSON?", "cli-reference.md"),
        ("Which command compiles manifests to artifacts?", "cli-reference.md"),
        ("Which command starts the development server?", "cli-reference.md"),
        ("Which option controls reload behavior in aq run?", "cli-reference.md"),
        ("Which command starts the production server?", "cli-reference.md"),
        ("What are the default timeout values for aq serve?", "cli-reference.md"),
        ("Which command freezes generated artifacts?", "cli-reference.md"),
        ("Which command updates manifests with auto-discovered resources?", "cli-reference.md"),
        ("Which command shows compiled routes?", "cli-reference.md"),
        ("Which command shows the DI graph?", "cli-reference.md"),
        ("Which command diagnoses workspace issues?", "cli-reference.md"),
        ("Which command generates a TypeScript client from WebSocket artifacts?", "cli-reference.md"),
        ("What phases does AquiliaRuntime move through?", "runtime-lifecycle.md"),
        ("What does configure() do before loading config?", "runtime-lifecycle.md"),
        ("What does discover() import and rebuild?", "runtime-lifecycle.md"),
        ("What does startup() initialize before request handling?", "runtime-lifecycle.md"),
        ("What endpoint is served before normal routing in handle_http?", "runtime-lifecycle.md"),
        ("How are HEAD requests handled in HTTP flow?", "runtime-lifecycle.md"),
        ("What does handle_websocket() delegate to?", "runtime-lifecycle.md"),
        ("What does graceful_shutdown() do to in-flight requests?", "runtime-lifecycle.md"),
        ("What is Aquilia described as?", "architecture.md"),
        ("What is the relationship between Workspace and Module?", "architecture.md"),
        ("What does AppManifest contain?", "architecture.md"),
        ("Which routes are registered when docs are enabled?", "architecture.md"),
        ("What does ASGIAdapter receive?", "architecture.md"),
        ("Which component executes the final HTTP handler?", "architecture.md"),
        ("What Python version is required?", "installation.md"),
        ("What is the console script?", "installation.md"),
        ("What core dependencies are listed?", "installation.md"),
        ("Which extra adds redis[asyncio] for cache/socket backends?", "installation.md"),
        ("Which extras are compatibility aliases?", "installation.md"),
        ("What commands appear in the first workspace workflow?", "installation.md"),
        ("What is Aquilia MCP?", "mcp/README.md"),
        ("What is the canonical package name?", "mcp/README.md"),
        ("What quick-start command builds the index?", "mcp/README.md"),
        ("What quick-start command lists tools?", "mcp/README.md"),
        ("What should you do if the tool list is empty?", "mcp/troubleshooting.md"),
        ("What resource URI patterns does resources/read accept?", "mcp/troubleshooting.md"),
        ("What does the documentation README say the docs are generated from?", "README.md"),
        ("Which page is used for the full module index?", "README.md"),
        ("How many package modules are documented in the coverage report?", "documentation-coverage-report.md"),
        ("Which module is described as the root framework runtime files?", "module-index.md"),
        ("Which module is described as the manifest registry and runtime registry constructor?", "module-index.md"),
        ("Which module is described as the URL pattern grammar and matcher docs?", "module-index.md"),
    ]
]

In [245]:
retriever_with_mmr = create_ensembler(
search_kwargs={"k": 20, "lambda_mul": 0.5, "fetch_k": 30}
)

retriever_with_semantic = create_ensembler(
search_type="similarity",
search_kwargs={"k": 20},
weights=[0.8, 0.2]
)

retriever_with_contextual_compression_mmr = create_contextual_ensembler(
search_kwargs={"k": 20, "lambda_mul": 0.5, "fetch_k": 30},
similarity_threshold=0.3
)

retriever_with_contextual_compression_semantic = create_contextual_ensembler(
search_type="similarity",
search_kwargs={"k": 20},
similarity_threshold=0.3
)

print("MMR:", evaluate_rag(dataset=eval_dataset, ensembler=retriever_with_mmr))
print("Semantic:", evaluate_rag(dataset=eval_dataset, ensembler=retriever_with_semantic))
print("Compression + MMR:", evaluate_rag(dataset=eval_dataset, ensembler=retriever_with_contextual_compression_mmr))
print("Compression + Semantic:", evaluate_rag(dataset=eval_dataset, ensembler=retriever_with_contextual_compression_semantic))

MMR: {'hit@1': 0.18421052631578946, 'hit@3': 0.5394736842105263, 'hit@5': 0.7368421052631579, 'mrr': 0.39543128654970744}
Semantic: {'hit@1': 0.05263157894736842, 'hit@3': 0.4342105263157895, 'hit@5': 0.631578947368421, 'mrr': 0.2870566567934989}
Compression + MMR: {'hit@1': 0.39473684210526316, 'hit@3': 0.6447368421052632, 'hit@5': 0.75, 'mrr': 0.5389872919478185}
Compression + Semantic: {'hit@1': 0.39473684210526316, 'hit@3': 0.6447368421052632, 'hit@5': 0.75, 'mrr': 0.5416301169590645}


In [246]:
question = eval_dataset[0]["question"]

docs_before = create_ensembler(search_kwargs={"k": 20, "lambda_mul": 0.5, "fetch_k": 30}
).invoke(question)
docs_after = create_contextual_ensembler(similarity_threshold=0.3, search_kwargs={"k": 20, "lambda_mul": 0.5, "fetch_k": 30}
).invoke(question)

print("before:", len(docs_before))
print("after:", len(docs_after))

before: 23
after: 20


In [367]:
class State(TypedDict):
    messages: typing.Annotated[list, add_messages]

In [398]:
class AgentResponse(BaseModel):
    summary: str = ""
    answer: str = ""

    work_performed: bool = False

    files_created: list[str] = Field(default_factory=list)
    files_modified: list[str] = Field(default_factory=list)
    directories_created: list[str] = Field(default_factory=list)

    changes: list[str] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    next_steps: list[str] = Field(default_factory=list)

In [399]:
agent = create_agent(
    model = AIProviders.llm,
    tools=toolbox,
    checkpointer=checkpoints,
    response_format=AgentResponse,
    system_prompt = SystemMessage(
"""
You are Aquilia, an expert software engineering assistant specialized in the Aquilia framework and codebase.

Your responsibilities include:
- Answering questions.
- Retrieving documentation.
- Exploring repositories.
- Writing code.
- Modifying files.
- Creating project structure.
- Explaining implementation details.

==================================================
WORKSPACE RULES
==================================================

Perform all filesystem operations strictly inside the workspace/.

Never access files outside workspace/.

All tool paths are already relative to workspace/.

Correct:
- ls(".")
- ls("src")
- read("src/main.py")
- write("main.py")
- mkdir("app/services")

Incorrect:
- ls("workspace")
- read("workspace/src/main.py")
- write("workspace/main.py")

==================================================
AVAILABLE TOOLS
==================================================

Documentation
- rag_search

Filesystem
- ls
- glob

Search
- grep

Files
- read
- write
- edit
- mkdir

==================================================
TOOL USAGE POLICY
==================================================

Documentation Questions:
- Use rag_search for Aquilia-specific information.
- Never invent APIs or undocumented behavior.
- Prefer retrieved documentation over assumptions.

Repository Tasks:
- Inspect before changing.
- Use ls, glob, grep, and read to gather context.
- Never modify files blindly.

Code Changes:
- Use mkdir for missing directories.
- Use write for new files.
- Use edit for existing files.
- Preserve project conventions.
- Make the smallest correct change.

Feature Development:
1. Inspect structure.
2. Locate relevant files.
3. Read necessary files.
4. Create missing directories if needed.
5. Implement feature.
6. Verify consistency.
7. Summarize results.

==================================================
REASONING GUIDELINES
==================================================

Before making changes:

- Understand the request.
- Gather sufficient context.
- Inspect existing implementations.
- Avoid assumptions.

When uncertain:
- Use tools to verify.
- State uncertainty explicitly.

Do not fabricate:
- APIs
- Classes
- Functions
- Configurations
- Documentation

==================================================
RESPONSE FORMAT RULES
==================================================

You MUST ALWAYS return a valid AgentResponse.

Rules:

1. summary is REQUIRED.
   - Never return null.
   - Always provide a short description of the outcome.

2. answer should contain the detailed response.

3. work_performed:
   - true if files/directories were created, modified, or deleted.
   - false for explanations, documentation answers, and chat.

4. files_created:
   - Include every created file.

5. files_modified:
   - Include every modified file.

6. directories_created:
   - Include every created directory.

7. changes:
   - Include implementation details and important actions.

8. sources:
   - Include documentation references from rag_search when used.
   - Include repository files that were inspected when relevant.

9. warnings:
   - Include assumptions, limitations, or unresolved issues.

10. next_steps:
   - Include recommended follow-up actions when useful.

Never return null for any field.
Use empty strings or empty lists instead.

==================================================
TASK TYPE BEHAVIOR
==================================================

For documentation questions:
- Use rag_search.
- Populate sources.
- work_performed = false.

For repository exploration:
- Inspect repository using tools.
- Report findings.
- work_performed = false unless files changed.

For coding tasks:
- Inspect first.
- Implement changes.
- Record affected files.
- work_performed = true.

For general programming questions:
- Answer directly.
- Do not use tools unless necessary.

==================================================
GOAL
==================================================

Act as a senior Aquilia engineer capable of:

- Understanding documentation.
- Navigating large codebases.
- Designing maintainable solutions.
- Implementing production-quality code.
- Safely modifying repositories.
- Explaining technical decisions clearly.

Always prioritize correctness, verification, and maintainability.
"""
    )
)

In [402]:
graph_image = agent.get_graph().draw_mermaid_png()
with open("agent_graph.png", "wb") as f:
    f.write(graph_image)

In [400]:
agent_response = agent.invoke(
    input = {
        "messages": [
            HumanMessage(
    """
    Create a main.py and write a reverse linked list code
"""
)
        ]
    },
    config=config
)
print(agent_response["structured_response"])

summary='Created `main.py` with an iterative reverse linked list implementation.' answer='Created `main.py` with:\n- `ListNode` class\n- `reverse_linked_list(head)` to reverse a singly linked list\n- helpers to build and convert the list for testing\n\n```python\nclass ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\n    def __repr__(self):\n        return f"ListNode(val={self.val})"\n\n\ndef reverse_linked_list(head: ListNode | None) -> ListNode | None:\n    """Reverse a singly linked list iteratively."""\n    prev = None\n    curr = head\n\n    while curr is not None:\n        nxt = curr.next\n        curr.next = prev\n        prev = curr\n        curr = nxt\n\n    return prev\n\n\ndef build_linked_list(values: list[int]) -> ListNode | None:\n    head = None\n    tail = None\n    for v in values:\n        node = ListNode(v)\n        if head is None:\n            head = tail = node\n        else:\n            tail.next = node\n 